# PhysioCoach-NLP: train Flan-T5 on Google Colab

Run this notebook on a **GPU runtime**: `Runtime -> Change runtime type -> T4 GPU`.

It builds the dataset, fine-tunes Flan-T5, evaluates it against the template baseline,
and gives you a zip of the trained model to copy into `models/flan-t5-coach` on your laptop.

In [ ]:
!nvidia-smi

## 1. Get the code
Upload `physiocoach-nlp.zip` with the **Files** panel on the left (folder icon), wait until the upload finishes, then run the next cell.
If you have already pushed the project to GitHub, use the commented `git clone` line instead.


In [ ]:
!unzip -q -o physiocoach-nlp.zip
# !git clone https://github.com/MMujtabaX/PhysioCoach-NLP.git physiocoach-nlp
%cd physiocoach-nlp
!ls

## 2. Install

In [ ]:
!pip install -q -e ".[ml,eval]"

## 3. (Optional) Save everything to Google Drive
Colab deletes files when the session ends. Mount Drive if you want the model to survive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 4. Build the dataset

In [ ]:
!python scripts/build_dataset.py --out data

## 5. Fine-tune Flan-T5
`flan-t5-small` takes roughly 15-25 minutes on a T4. Try `google/flan-t5-base` afterwards
(`--out models/flan-t5-coach-base`) and compare.

In [ ]:
!python scripts/train_t5.py --data data --model google/flan-t5-small --out models/flan-t5-coach --epochs 6

## 6. Evaluate against the template baseline

In [ ]:
!PHYSIOCOACH_T5_DIR=models/flan-t5-coach python scripts/evaluate.py --test data/test.jsonl --backends template,t5 --limit 300
!cat results/metrics.md

## 7. (Optional) Export to ONNX for faster CPU inference

In [ ]:
!pip install -q "optimum[onnxruntime]"
!optimum-cli export onnx --model models/flan-t5-coach --task text2text-generation-with-past models/flan-t5-coach-onnx

## 8. Download the trained model and results

In [ ]:
!zip -qr flan-t5-coach.zip models/flan-t5-coach -x "models/flan-t5-coach/checkpoints/*"
!zip -qr results.zip results data/stats.json
from google.colab import files
files.download("flan-t5-coach.zip")
files.download("results.zip")